In [1]:
from flax import nnx
import jax
from model import Encoder_Decoder
import orbax.checkpoint as ocp

In [27]:
rngs = nnx.Rngs(params=0)
checkpointer = ocp.StandardCheckpointer()
checkpoint_dir = "/home/roman/learning-ml/2802ict-assignments/assignment_01/maze_src/cnn_heuristic/checkpoints/save-no-normalisation"

In [ ]:
abstract_model: Encoder_Decoder = nnx.eval_shape(lambda: Encoder_Decoder(nnx.Rngs(0)))
graphdef, abstract_state = nnx.split(abstract_model)
restored_state = checkpointer.restore(checkpoint_dir, abstract_state)
model = nnx.merge(graphdef, restored_state)
model.eval() # recursively sets deterministic=True and use_running_average=True

@nnx.jit
def apply(m: Encoder_Decoder, x):
    return m(x)

/home/roman/learning-ml/.venv/lib/python3.12/site-packages/orbax/checkpoint/_src/serialization/type_handlers.py:1251: UserWarning:

Sharding info not provided when restoring. Populating sharding info from sharding file. Please note restoration time will be slightly increased due to reading from file. Note also that this option is unsafe when restoring on a different topology than the checkpoint was saved with.



In [65]:
from dataset import SingleFolderDataset, show_image
import jax_dataloader as jdl
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

In [67]:
pt_ds_train = SingleFolderDataset("./motion_planning_datasets/mazes/train")
dl_train = jdl.DataLoader(pt_ds_train, 'pytorch', batch_size=32, shuffle=True)

In [71]:

X, y = next(iter(dl_train))
# show_image(X[0])
# show_image(y[0])

# px.imshow(X[0][:,:,0])

fig = make_subplots(rows=1, cols=3, subplot_titles=('obstacle map', 'euclidian transform map', 'euclidian distance map'), horizontal_spacing=0.03)

fig.add_trace(
    go.Heatmap(z=X[0][...,0], colorscale='Viridis',
               colorbar=dict(orientation='h',y=-0.4, x=0.15, len=0.3, lenmode='fraction')),
    row=1, col=1
)
fig.add_trace(
    go.Heatmap(z=X[0][...,1], colorscale='Viridis',
               colorbar=dict(orientation='h', y=-0.4, x=0.5,  len=0.3, lenmode='fraction')),
    row=1, col=2
)
fig.add_trace(
    go.Heatmap(z=X[0][...,2], colorscale='Viridis',
               colorbar=dict(orientation='h', y=-0.4, x=0.85, len=0.3, lenmode='fraction')),
    row=1, col=3
)

fig.update_layout(width=1000, height=350, margin=dict(t=40))
fig.show()


# y_pred = model(X)

# show_image(y_pred[0])


In [72]:
y_pred = apply(model, X)

# print(y_pred[0])

# px.imshow(y[0][...,0])

# px.imshow(y_pred[0][...,0])

fig = make_subplots(rows=1, cols=3, subplot_titles=('label', 'label mask', 'prediction'), horizontal_spacing=0.03)

fig.add_trace(
    go.Heatmap(z=y[0][...,0], colorscale='Viridis',
               colorbar=dict(orientation='h',y=-0.4, x=0.15, len=0.3, lenmode='fraction')),
    row=1, col=1
)
fig.add_trace(
    go.Heatmap(z=y[0][...,1], colorscale='Viridis',
               colorbar=dict(orientation='h', y=-0.4, x=0.5,  len=0.3, lenmode='fraction')),
    row=1, col=2
)
fig.add_trace(
    go.Heatmap(z=y_pred[0][...,0], colorscale='Viridis',
               colorbar=dict(orientation='h', y=-0.4, x=0.85, len=0.3, lenmode='fraction')),
    row=1, col=3
)

fig.update_layout(width=1000, height=350, margin=dict(t=40))
fig.show()